## DATA 37711 Homework 3
### Glen Morgenstern

## Problem 1
### ai.
Poisson gives us matching support. Y takes in values {0, 1, 2, ...} and Poisson is supported on exactly those values. It's also the canoncial model for count data and has a single parameter so we only need to model the conditional mean.

### aii.
The Poisson rate $\mu$ must be strictly positive, but $f(X) = \sum_j \lambda_j h_j(X)$ is a sum of regression trees and can take any real value. The link function must therefore map $\mathbb{R} \to (0,\infty)$. The exponential does this, and it is the canonical link for the Poisson.

### 1b.
Under the model, $Y \mid X \sim \text{Poi}(\mu)$ with $\mu = \exp(f(X))$, so
$$
P(Y = y \mid X = x) = \frac{\mu^{y} e^{-\mu}}{y!} = \frac{\exp\!\big(y f(x)\big)\exp\!\big(-e^{f(x)}\big)}{y!}.
$$
The negative log likelihood is
$$
-\log P(Y \mid X) = e^{f(X)} - Y f(X) + \log(Y!).
$$
The term $\log(Y!)$ does not depend on $f$, so it can be dropped. The per-example loss is therefore
$$
L\big(f(x), Y\big) = e^{f(x)} - Y f(x),
$$
and we fit $f$ by minimizing the empirical risk
$$
\hat{R}(f) = \frac{1}{n}\sum_{i=1}^{n}\Big( e^{f(x_i)} - y_i f(x_i) \Big).
$$
This loss is convex in $f(x)$.

### 1c.
Differentiating per-example loss $L(f(x), Y) = e^{f(x)} - Y f(x)$ with respect to $f(x)$ gives
$$
\frac{dL}{df(x)} = e^{f(x)} - Y.
$$
This is the predicted mean $\hat{\mu} = e^{f(x)}$ minus the observed count $Y$. The negative gradient, $Y - e^{f(x)}$, is therefore the residual on the mean scale.

### 1d.
First order boosting is gradient descent in function sapce. We build f greedily, one tree at a time.

1. Initialize with the best constant, $f_0(x) = \log \bar{y}$, where $\bar{y} = \frac{1}{n}\sum_i y_i$ (this minimizes $\sum_i e^{c} - y_i c$ over constants $c$)
2. For $j = 1, \dots, J$:
  - Compute negative gradients at current model:
$$
r_i^{(j)} = -\frac{dL}{df(x_i)}\bigg|_{f = f_{j-1}} = y_i - e^{f_{j-1}(x_i)}.
$$
  - Fit a regression tree $h_j$ to $\{(x_i, r_i^{(j)})\}_{i=1}^n$ by ordinary least squares, i.e.
$$
h_j = \arg\min_{h} \sum_{i=1}^n \big( r_i^{(j)} - h(x_i) \big)^2 .
$$
  - Choose the step size $\lambda_j$, either as a fixed small learning rate or by line search
$$
\lambda_j = \arg\min_{\lambda} \sum_{i=1}^n L\big(f_{j-1}(x_i) + \lambda h_j(x_i),\, y_i\big).
$$
  - Update $f_j(x) = f_{j-1}(x) + \lambda_j h_j(x)$.

3. The final model is $f(x) = f_0(x) + \sum_{j=1}^J \lambda_j h_j(x)$, with predicted mean $\hat{\mu}(x) = e^{f(x)}$.